# LangGraph Telecom Lab Notebook (40 Minutes)

## Topic: State + Reducers + Memory + HITL Approval Workflow

This notebook is a **single consolidated hands-on lab** for an **IBM Telecom Agentic AI Team**.

### What you will build
A **telecom incident approval workflow** in LangGraph with:

- **State object**
- **Reducers**
- **Short-term memory**
- **Long-term memory**
- **HITL (Human-in-the-Loop)** approval
- **Approval workflow**
- **Memory save and retrieval**

---

## Lab Flow (40 Minutes)

| Time | Section |
|---|---|
| 5 min | Setup + concepts |
| 5 min | Step 1: State object |
| 5 min | Step 2: Reducers |
| 10 min | Step 3: Build first approval graph |
| 7 min | Step 4: Add short-term memory |
| 5 min | Step 5: Add long-term memory |
| 3 min | Step 6: Participant exercises |

---

## Telecom Use Case

Customer complaint:

> "My mobile internet is very slow in Pune since morning."

The workflow will:

1. collect the incident
2. diagnose the issue
3. propose an action
4. pause for human approval
5. continue only after approval
6. save final resolution memory

---

## Prerequisites

Install:

```bash
pip install langgraph
```

If your environment does not have `langgraph` installed, run the install cell below.

In [ ]:
# Optional install cell
# Uncomment if needed

# %pip install langgraph

## Step 1 — Imports

In [ ]:
from typing import TypedDict, Annotated, List, Dict, Any
from operator import add

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt, Command

## Step 2 — Understand the State Object

A **state object** is the shared data that moves through the graph.

In this telecom lab, the state holds:

- incident details
- diagnosis
- proposed action
- approval decision
- audit log
- memory notes

In [ ]:
class TelecomApprovalState(TypedDict, total=False):
    incident_id: str
    customer_id: str
    complaint: str
    location: str
    severity: str

    diagnosis: str
    proposed_action: str

    approval_required: bool
    approved: bool
    approver_comment: str

    # Reducer-enabled logs
    audit_log: Annotated[List[str], add]
    memory_notes: Annotated[List[str], add]

### Quick Check
Let us create a sample state object.

In [ ]:
sample_state: TelecomApprovalState = {
    "incident_id": "INC001",
    "customer_id": "CUST_1001",
    "complaint": "My mobile internet is very slow in Pune since morning.",
    "audit_log": [],
    "memory_notes": []
}

sample_state

## Step 3 — Reducers

A **reducer** controls how values are combined in state.

In this notebook we use:

```python
Annotated[List[str], add]
```

That means when multiple nodes return:

```python
{"audit_log": ["some message"]}
```

LangGraph will **append** messages instead of overwriting them.

### Why reducers matter
Without reducers:
- later node updates can replace earlier values

With reducers:
- messages/logs accumulate naturally

In [ ]:
# Demonstration: reducers concept using plain Python lists
a = ["Incident collected"]
b = ["Diagnosis completed"]
print(a + b)

## Step 4 — Mock Telecom Backend Functions

These simulate telecom systems like:
- CRM
- network diagnosis
- decision engine

In [ ]:
def collect_incident_details(incident_id: str, complaint: str) -> Dict[str, Any]:
    return {
        "incident_id": incident_id,
        "location": "Pune",
        "severity": "high"
    }


def diagnose_incident(location: str, complaint: str) -> Dict[str, Any]:
    # Simulated diagnosis logic
    return {
        "diagnosis": "Likely local tower congestion during peak hours.",
        "proposed_action": "Escalate to network operations and notify enterprise support.",
        "approval_required": True
    }

## Step 5 — Create Nodes

Each node:
- reads state
- performs one task
- returns updates to state

In [ ]:
def collect_incident_node(state: TelecomApprovalState):
    details = collect_incident_details(
        incident_id=state["incident_id"],
        complaint=state["complaint"]
    )
    return {
        "location": details["location"],
        "severity": details["severity"],
        "audit_log": [f"Incident collected for {details['location']} with severity {details['severity']}"]
    }


def diagnose_node(state: TelecomApprovalState):
    result = diagnose_incident(
        location=state["location"],
        complaint=state["complaint"]
    )
    return {
        "diagnosis": result["diagnosis"],
        "proposed_action": result["proposed_action"],
        "approval_required": result["approval_required"],
        "audit_log": ["Diagnosis completed"]
    }

## Step 6 — Add HITL Node

This is the **human-in-the-loop** step.

If approval is required, the workflow pauses here using:

```python
interrupt(...)
```

Later, we resume the graph with:

```python
Command(resume={...})
```

In [ ]:
def approval_node(state: TelecomApprovalState):
    if state.get("approval_required", False):
        human_decision = interrupt({
            "message": "Approval required for high-impact telecom action",
            "incident_id": state["incident_id"],
            "diagnosis": state["diagnosis"],
            "proposed_action": state["proposed_action"]
        })
        return {
            "approved": human_decision.get("approved", False),
            "approver_comment": human_decision.get("comment", ""),
            "audit_log": [f"Approval decision received: {human_decision}"]
        }
    return {
        "approved": True,
        "audit_log": ["Approval not required"]
    }

## Step 7 — Execution Node

This node executes the proposed action only if approval is granted.

In [ ]:
def execute_action_node(state: TelecomApprovalState):
    if state.get("approved"):
        return {
            "audit_log": [f"Executed action: {state['proposed_action']}"],
            "memory_notes": ["Approved telecom action executed successfully"]
        }
    return {
        "audit_log": ["Action not executed because approval was denied"],
        "memory_notes": ["Approval denied; no telecom action executed"]
    }

## Step 8 — Long-Term Memory Node

This node writes final incident details into long-term memory.

We will use an **InMemoryStore** for the lab.

In a real project, this could be replaced with a durable store.

In [ ]:
def save_memory_node(state: TelecomApprovalState, config, store):
    namespace = ("telecom_memories", state["incident_id"])
    key = "final_resolution"

    memory_doc = {
        "incident_id": state["incident_id"],
        "customer_id": state.get("customer_id", ""),
        "location": state.get("location", ""),
        "severity": state.get("severity", ""),
        "diagnosis": state.get("diagnosis", ""),
        "proposed_action": state.get("proposed_action", ""),
        "approved": state.get("approved", False),
        "approver_comment": state.get("approver_comment", "")
    }

    store.put(namespace, key, memory_doc)

    return {
        "audit_log": ["Saved long-term memory for incident"],
        "memory_notes": ["Incident memory stored"]
    }

## Step 9 — Build the Graph

Flow:

```text
collect_incident
   ↓
diagnose
   ↓
approval (HITL interrupt)
   ↓
execute_action
   ↓
save_memory
   ↓
END
```

In [ ]:
builder = StateGraph(TelecomApprovalState)

builder.add_node("collect_incident", collect_incident_node)
builder.add_node("diagnose", diagnose_node)
builder.add_node("approval", approval_node)
builder.add_node("execute_action", execute_action_node)
builder.add_node("save_memory", save_memory_node)

builder.set_entry_point("collect_incident")
builder.add_edge("collect_incident", "diagnose")
builder.add_edge("diagnose", "approval")
builder.add_edge("approval", "execute_action")
builder.add_edge("execute_action", "save_memory")
builder.add_edge("save_memory", END)

## Step 10 — Add Short-Term Memory and Long-Term Memory

### Short-term memory
We use a **checkpointer** so the graph can pause and resume.

### Long-term memory
We use a **store** so the graph can keep durable facts across sessions.

In [ ]:
checkpointer = InMemorySaver()
store = InMemoryStore()

graph = builder.compile(checkpointer=checkpointer, store=store)

## Step 11 — Run the Workflow Until Pause

This first run will pause at the HITL approval step.

In [ ]:
config = {"configurable": {"thread_id": "incident-thread-001"}}

initial_input = {
    "incident_id": "INC001",
    "customer_id": "CUST_1001",
    "complaint": "My mobile internet is very slow in Pune since morning.",
    "audit_log": [],
    "memory_notes": []
}

paused_result = graph.invoke(initial_input, config=config)
paused_result

### What happened?
The graph ran through:
- `collect_incident`
- `diagnose`

Then it paused at:
- `approval`

Because `interrupt(...)` was triggered.

This is the HITL point.

## Step 12 — Resume with Human Approval

Now simulate a human approving the action.

In [ ]:
approved_result = graph.invoke(
    Command(resume={"approved": True, "comment": "Approved by NOC lead"}),
    config=config
)

approved_result

## Step 13 — Inspect Final State

The state should now contain:
- diagnosis
- approved = True
- approver comment
- appended audit log
- memory notes

In [ ]:
print("Diagnosis:", approved_result.get("diagnosis"))
print("Approved:", approved_result.get("approved"))
print("Approver Comment:", approved_result.get("approver_comment"))
print("\nAudit Log:")
for item in approved_result.get("audit_log", []):
    print("-", item)

print("\nMemory Notes:")
for item in approved_result.get("memory_notes", []):
    print("-", item)

## Step 14 — Read Long-Term Memory

The workflow saved incident resolution into long-term memory.

Let us retrieve it.

In [ ]:
saved_memory = store.get(("telecom_memories", "INC001"), "final_resolution")
saved_memory

## Step 15 — Denial Scenario

Now test what happens when approval is denied.

Use a new thread ID so the second run is independent.

In [ ]:
config_denied = {"configurable": {"thread_id": "incident-thread-002"}}

# Run until pause
_ = graph.invoke(
    {
        "incident_id": "INC002",
        "customer_id": "CUST_2002",
        "complaint": "Voice quality is very poor in Pune.",
        "audit_log": [],
        "memory_notes": []
    },
    config=config_denied
)

# Resume with denial
denied_result = graph.invoke(
    Command(resume={"approved": False, "comment": "Need more evidence before escalation"}),
    config=config_denied
)

denied_result

## Step 16 — Compare Approval vs Denial

This helps participants understand how HITL changes the flow outcome.

In [ ]:
print("=== APPROVED INCIDENT ===")
print("Approved:", approved_result.get("approved"))
print("Audit Log:")
for item in approved_result.get("audit_log", []):
    print("-", item)

print("\n=== DENIED INCIDENT ===")
print("Approved:", denied_result.get("approved"))
print("Audit Log:")
for item in denied_result.get("audit_log", []):
    print("-", item)

## Step 17 — Summary of Concepts

### State object
Holds workflow data across all steps.

### Reducers
Append or merge state values like logs and messages.

### Short-term memory
Provided by the **checkpointer**.
Useful for:
- pause/resume
- thread continuity
- intermediate state persistence

### Long-term memory
Provided by the **store**.
Useful for:
- durable telecom incident history
- reusable customer facts
- cross-session knowledge

### HITL
Pauses execution for human review or approval.
Useful for:
- escalations
- customer credits
- risky network actions
- compliance-sensitive operations

## Step 18 — Participant Exercises

### Exercise 1
Add a new state field:

```python
resolution_status: str
```

Update `execute_action_node` so it sets:
- `"resolution_status": "approved_and_executed"`
- or `"resolution_status": "denied"`

---

### Exercise 2
Add another HITL gate:
Require approval only when:
- severity is `"high"`

---

### Exercise 3
Store one more long-term memory field:
- timestamp
- region
- escalation level

---

### Exercise 4
Add one more reducer-enabled field:

```python
notifications: Annotated[List[str], add]
```

Use it to log:
- SMS sent
- NOC team notified
- enterprise support informed

## Step 19 — Final Takeaway

You have now built a **LangGraph telecom approval workflow** with:

- state
- reducers
- short-term memory
- long-term memory
- HITL approval
- persistent resolution memory

This is a strong foundation for:
- telecom incident response agents
- escalation approval workflows
- enterprise AI operations systems